# Time-domain: Trapping Square

In [1]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw
import numpy as np

In [13]:
# domain parameters
Rout = 2
RScat = 1
DScat = 0.1
DHole = 0.2
R=1.5

#Method parameters
maxh = 0.1  
order = 4

#Problem parameters
k = 10.8
theta = 0
omega = 8.1

xOffset = 0.5

In [14]:
def create_geo(Rout = Rout, RScat = RScat, DScat = DScat, DHole = DHole, R = R, xOffset = xOffset,draw = True):
    domain = Circle((0, 0), Rout).Face()
    domain.edges.name = 'outer'
    domain.faces.name = 'outer'
    
    inner = Circle((0, 0), R).Face()
    inner.edges.name = 'interface'
    inner.faces.name = 'inner'

    scatterer = (
            MoveTo(-RScat/2+xOffset,-RScat/2).Rectangle(RScat,RScat).Face()
            -MoveTo(-RScat/2+DScat/2+xOffset,-RScat/2+DScat/2).Rectangle(RScat-DScat,RScat-DScat).Face()
            -MoveTo(-RScat/2-DScat+xOffset,-DHole/2).Rectangle(2*DScat,DHole).Face()
        )
    scatterer.edges.name = 'scatterer'

    scattererdom = MoveTo(-RScat/2+xOffset,-RScat/2).Rectangle(RScat,RScat).Face()-scatterer
    scattererdom.faces.name = 'scatterer'
    


    if draw:
        Draw(Glue([domain-inner,inner-scattererdom-scatterer, scattererdom-scatterer]))
    geo = OCCGeometry(Glue([domain-inner,inner-scattererdom-scatterer, scattererdom]), dim=2)
    return geo

In [15]:
geo = create_geo(draw = False)
mesh = Mesh(geo.GenerateMesh(maxh=maxh))
mesh.Curve(order)
Draw(mesh)
print(mesh.GetMaterials(),mesh.GetBoundaries())

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

('outer', 'inner', 'scatterer') ('outer', 'interface', 'default', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer', 'scatterer')


In [16]:
def get_bilinear_forms(maxh = maxh, order = order):
    geo = create_geo(Rout = Rout, RScat = RScat, DScat = DScat, DHole = DHole, R = R, draw = False)
    mesh = Mesh(geo.GenerateMesh(maxh=maxh))
    mesh.Curve(order)

    fes = H1(mesh,order = order, dirichlet = "scatterer")

    w,w_ = fes.TnT()

    rho = sqrt(x**2+y**2)

    #Omega = CF([(Rout-rho)/(Rout-R),1,1])
    Omega = mesh.MaterialCF({"outer": (Rout-rho)/(Rout-R)},default=1)

    DOmega = Omega.Diff(rho)
    L = Omega-rho*DOmega
    G = Omega**2/L

    #H = CF([1 - G,0,0]) #characteristic preserving
    H = mesh.MaterialCF({"outer": 1-G},default=0) #characteristic preserving
    Mu = 1+H  #(1+H^2)/G, simplifies to 1+H for characteristic preserving


    vx = CF((x,y))

    Pi = OuterProduct(vx,vx)/rho**2 #projection onto radial vector
    Pi_perp = Id(2)-Pi              #projection onto tangential vector

    ### k**2
    M_ = Mu*w*w_*dx

    ### 1j*k
    C_ = (
        -H/rho*w*InnerProduct(vx,grad(w_))*dx
        +H/rho*w_*InnerProduct(vx,grad(w))*dx
        +w_*w*ds('outer')
        )

    ## 1
    K_ = (
        -grad(w)*( (G*Pi+L*Pi_perp)*grad(w_))*dx
        -Omega/L/2/rho*DOmega*InnerProduct(grad(w),vx)*w_*dx
        -Omega/L/2/rho*DOmega*InnerProduct(grad(w_),vx)*w*dx
        -1/L/4*DOmega**2*w*w_*dx
        )


    return M_, C_, K_,fes

In [17]:
def compute_inner_mag(gfu,gfup):
    u,v = gfu.space.TnT()
    mass_scat = BilinearForm(u*v*dx('scatterer'),check_unused=False).Assemble().mat
    mass_inner = BilinearForm(u*v*dx('inner|scatterer'),check_unused=False).Assemble().mat
    
    stiffness_scat = BilinearForm(grad(u)*grad(v)*dx('scatterer'),check_unused=False).Assemble().mat
    stiffness_inner = BilinearForm(grad(u)*grad(v)*dx('inner|scatterer'),check_unused=False).Assemble().mat
    
    norm_inner = sqrt(gfup.vec.InnerProduct(mass_inner*gfup.vec)+gfu.vec.InnerProduct(stiffness_inner*gfu.vec)).real
    norm_scat = sqrt(gfup.vec.InnerProduct(mass_scat*gfup.vec)+gfu.vec.InnerProduct(stiffness_scat*gfu.vec)).real

    return norm_inner, norm_scat 

In [18]:
def get_newmark_beta_matrices(M_,C_,K_,gamma, beta, freedofs):
    M = BilinearForm(M_).Assemble().mat
    C = BilinearForm(C_).Assemble().mat
    K = BilinearForm((-1)*K_).Assemble().mat
    Sinv = BilinearForm(M_+gamma*dt*C_-beta*dt**2*K_).Assemble().mat.Inverse(freedofs = freedofs)
    Minv = M.Inverse(freedofs = freedofs)
    return M,C,K,Sinv,Minv

In [19]:
dt = 2e-2

beta = 1/4
gamma = 1/2
M_, C_, K_, fes = get_bilinear_forms(maxh = maxh, order = order)
M, C, K, Sinv, Minv = get_newmark_beta_matrices(M_,C_,K_,gamma, beta,fes.FreeDofs())

In [32]:
def run_simulation(omega):
    w0 = exp(-((x+0.8)**2+(y+0.8)**2)*30)
        
    
    t0=10
    forcefact=10000
    force = LinearForm(forcefact*w0*fes.TestFunction()*dx).Assemble().vec
    
    gfw = GridFunction(fes)
    #gfw.Set(w0)
    gfw.Set(0.)
    
    gfw_ = GridFunction(fes)
    gfw_.Set(0.)
    
    gfw__ = GridFunction(fes)
    gfw__.Set(0.)
    
    gfw__.vec.data = -Minv@K*gfw.vec-Minv@C*gfw_.vec
    
    tmp = gfw__.vec.CreateVector()
    
    #draw with webgui
    #scene = Draw(gfw,min=-0.1,max=0.1)
    gf_anim = GridFunction(fes,multidim=0)

    drawtimes = int(0.2/dt)
    t=0.
    tend = 10
    
    #with open("output/"+filename+".out",'w') as f:
    #    f.write("# Rout = {}, RScat = {}, DScat = {}, DHole = {}, R = {}, order = {}, theta = {}\n".format(Rout, RScat, DScat, DHole, R, order, theta))
    #    f.write("# t\t norm inner \t norm scatterer\n")
    
    inner_norms = []
    
    i = 0
    #newmark beta
    while t<tend:
        if not i%drawtimes:
            #scene.Redraw()
            norm_inner,norm_scat = compute_inner_mag(gfw,gfw_)
            print("t = {}, norm inner = {}, norm_scat = {}".format(t,norm_inner,norm_scat),end='\r')
            gf_anim.AddMultiDimComponent(gfw.vec)
            #with open("output/"+filename+".out",'a') as f:
            #    f.write("{}\t{}\t{}\n".format(t,norm_inner,norm_scat))
        tmp.data = gfw__.vec
        gfw__.vec.data = -Sinv @ K*gfw.vec-Sinv@C*gfw_.vec-dt*Sinv@K*gfw_.vec+(gamma-1)*dt*Sinv@C*tmp+(beta-1/2)*dt**2*Sinv@K*tmp+(t<t0)*sin(omega*(t+dt))*force
        gfw.vec.data += dt*gfw_.vec+(1/2-beta)*dt**2*tmp+beta*dt**2*gfw__.vec
        gfw_.vec.data += (1-gamma)*dt*tmp+gamma*dt*gfw__.vec
        i+=1
        t+=dt

    print("")
    print("simulation done")
    print("")
    return gf_anim


In [35]:
gf_anim = run_simulation(10.8)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
t = 9.999999999999876, norm inner = 4.307206618086548, norm_scat = 0.9888496062571083655
simulation done

t = 9.999999999999876, norm inner = 4.686426464479681, norm_scat = 0.925047717051700316106
simulation done



In [36]:
Draw(gf_anim,min=-0.1,max=0.1,animate=True)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene